# TTRPG LLM Playground - Train/Eval Only

This notebook runs training and evaluation assuming synthetic data already exists.


In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 2. Setup Workspace
import os

# Define paths
REPO_URL = "https://github.com/riefer02/trpg-llm-playground.git" # Updated automatically
PROJECT_DIR = "/content/trpg-llm-playground"

# Clone or pull repo
if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL}
else:
    %cd /content/trpg-llm-playground
    !git pull
%cd {PROJECT_DIR}


In [ ]:
# 3. Install Dependencies (Official Unsloth method for Colab)
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Official Unsloth Colab install - specific versions for compatibility
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer psutil
    !pip install --no-deps unsloth

# Install remaining project dependencies
!pip install openai PyYAML tqdm pymupdf pydantic


In [ ]:
# 4. Run Training
# Ensure config/rpg_finetune.yaml points to the dataset produced earlier.
!python -m src.training.finetune_lora --config config/rpg_finetune.yaml


In [ ]:
# 5. Evaluate (Optional)
# !python -m src.training.evaluate --config config/rpg_finetune.yaml


In [ ]:
# 6. Login to Hugging Face
# 
# Option A: Use Colab Secrets (recommended)
#   - Add 'HF_TOKEN' to your Colab Secrets (Key icon in left sidebar)
#   - Get your token from: https://huggingface.co/settings/tokens
#
# Option B: Interactive login (will prompt for token)

from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("✅ Logged in via Colab Secrets")
except Exception:
    print("HF_TOKEN not found in secrets. Starting interactive login...")
    login()  # Will prompt for token input


In [ ]:
# 7. Push Model to Hugging Face Hub
#
# This uploads your fine-tuned model so you can:
# - Use it in HF Spaces
# - Share it with others
# - Download it later for local inference
#
# CONFIGURE THESE:
HF_USERNAME = "your-username"  # Your HF username
MODEL_NAME = "lancer-rules-7b-lora"  # Name for your model
PUSH_MERGED = True  # True = merge LoRA into base (larger, easier to use)
                    # False = push LoRA adapter only (smaller, requires base model)

import yaml
import os

# Load config to get output path
with open("config/rpg_finetune.yaml", "r") as f:
    cfg = yaml.safe_load(f)

path_vars = {
    "project_name": cfg.get("project_name", "default"),
    "dataset_tag": cfg.get("dataset_tag", "v1")
}
output_dir = cfg["training"]["output_dir"].format(**path_vars)

print(f"Loading model from: {output_dir}")

# Reload the trained model
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=output_dir,
    max_seq_length=cfg["model"].get("max_seq_length", 4096),
    load_in_4bit=True,
)

# Push to Hub
repo_id = f"{HF_USERNAME}/{MODEL_NAME}"

if PUSH_MERGED:
    print(f"Merging LoRA and pushing to: {repo_id}")
    model.push_to_hub_merged(
        repo_id,
        tokenizer,
        save_method="merged_16bit",
        private=False,
    )
    print(f"✅ Merged model pushed to: https://huggingface.co/{repo_id}")
else:
    print(f"Pushing LoRA adapter to: {repo_id}")
    model.push_to_hub(repo_id, tokenizer, private=False)
    print(f"✅ LoRA adapter pushed to: https://huggingface.co/{repo_id}")
    print(f"Note: Users will need base model '{cfg['model']['base_model']}' to use this.")


In [ ]:
# 8. (Optional) Export to GGUF for Ollama/Local Use
#
# This converts your model to GGUF format for:
# - Running locally with Ollama, LM Studio, llama.cpp
# - Smaller file size with quantization
#
# Skip this cell if you only need the HF version.

EXPORT_GGUF = False  # Set True to export
QUANTIZATION = "q4_k_m"  # Options: f16, q8_0, q4_k_m, q4_0 (smaller = faster but lower quality)

if EXPORT_GGUF:
    print(f"Exporting to GGUF with {QUANTIZATION} quantization...")
    
    # Unsloth has built-in GGUF export
    gguf_path = f"/content/drive/MyDrive/llm_experiments/outputs/{MODEL_NAME}-{QUANTIZATION}.gguf"
    
    model.save_pretrained_gguf(
        gguf_path.replace(".gguf", ""),
        tokenizer,
        quantization_method=QUANTIZATION,
    )
    
    print(f"✅ GGUF exported to: {gguf_path}")
    print(f"\nTo use with Ollama locally:")
    print(f"1. Download the .gguf file from Google Drive")
    print(f"2. Create a Modelfile:")
    print(f'   FROM ./{MODEL_NAME}-{QUANTIZATION}.gguf')
    print(f"3. Run: ollama create {MODEL_NAME} -f Modelfile")
    print(f"4. Run: ollama run {MODEL_NAME}")
else:
    print("Skipping GGUF export. Set EXPORT_GGUF = True to enable.")
